ถ้า error หรือติดอะไร ให้ถามต่อจากแชท https://chatgpt.com/s/t_69b06c8d55088191a279e7ba12fd2e71

In [ ]:
pip install ultralytics opencv-python tensorflow numpy pandas

### **Detector**

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO("models/best.pt")

def detect_cells(image):

    results = model(image)[0]

    boxes = []

    for box in results.boxes:

        x1,y1,x2,y2 = map(int,box.xyxy[0])
        conf = float(box.conf[0])

        boxes.append({
            "bbox":[x1,y1,x2,y2],
            "confidence":conf
        })

    return boxes

### **Classifier**

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

model = load_model("../models/classifier.h5")

CLASS_NAMES = ["normal","abnormal"]

def preprocess(img):

    img = cv2.resize(img,(224,224))
    img = img / 255.0
    img = np.expand_dims(img,0)

    return img


def classify(cell_img):

    inp = preprocess(cell_img)

    pred = model.predict(inp,verbose=0)[0]

    class_id = int(np.argmax(pred))

    return {
        "class":CLASS_NAMES[class_id],
        "confidence":float(pred[class_id])
    }

### **Model pipeline**

In [ ]:
import cv2
import os
import numpy as np

from detector import detect_cells
from classifier import classify

RESULT_FOLDER = "../results"

os.makedirs(RESULT_FOLDER,exist_ok=True)


def analyze_images(images):

    cell_results = []
    image_summary = []

    for image_id,image in enumerate(images):

        boxes = detect_cells(image)

        total_cells = 0
        abnormal_cells = 0

        abnormal_boxes = []

        for i,b in enumerate(boxes):

            x1,y1,x2,y2 = b["bbox"]

            crop = image[y1:y2,x1:x2]

            if crop.size == 0:
                continue

            total_cells += 1

            result = classify(crop)

            if result["class"] == "abnormal":

                abnormal_cells += 1
                abnormal_boxes.append((x1,y1,x2,y2))

            cell_results.append({
                "image_id":image_id,
                "cell_id":i,
                "bbox":b["bbox"],
                "predicted_class":result["class"],
                "confidence":result["confidence"]
            })

        vis = image.copy()

        for i,(x1,y1,x2,y2) in enumerate(abnormal_boxes):

            cv2.rectangle(vis,(x1,y1),(x2,y2),(0,0,255),2)

            label = f"Cell {i+1}"

            cv2.putText(
                vis,
                label,
                (x1, y1-5),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0,0,255),
                1,
                cv2.LINE_AA
            )

        path = f"{RESULT_FOLDER}/image_{image_id}.jpg"

        cv2.imwrite(path,vis)

        image_summary.append({
            "image_id":image_id,
            "total_cells":total_cells,
            "abnormal_cells":abnormal_cells,
            "image_path":path
        })

    return summarize(image_summary)


def summarize(image_summary):

    total_cells = sum(i["total_cells"] for i in image_summary)
    total_abnormal = sum(i["abnormal_cells"] for i in image_summary)

    suspicious = [i for i in image_summary if i["abnormal_cells"]>0]

    suspicious_sorted = sorted(
        suspicious,
        key=lambda x:x["abnormal_cells"],
        reverse=True
    )[:30]

    return {
        "total_cells_detected":total_cells,
        "total_suspicious_cells_detected":total_abnormal,
        "suspicious_images_detected":len(suspicious),
        "top_suspicious_images":suspicious_sorted
    }

### **FastAPI Backend**

In [ ]:
from fastapi import FastAPI,UploadFile,File
from typing import List
import numpy as np
import cv2

from model_pipeline import analyze_images

app = FastAPI()


@app.post("/analyze")
async def analyze(files: List[UploadFile] = File(...)):

    images = []

    for file in files:

        content = await file.read()

        arr = np.frombuffer(content,np.uint8)

        img = cv2.imdecode(arr,cv2.IMREAD_COLOR)

        images.append(img)

    result = analyze_images(images)

    return result